In [1]:
# 1. Import libraries
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [2]:
# 2. Load dataset
df = pd.read_csv("Telecom-Customer-Churn.csv")


In [3]:
# 3. Explore data
print(df.head())
df.info()
print(df.isnull().sum())


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [4]:
# 4. Handle missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())


In [5]:
# 5. Remove duplicates
df = df.drop_duplicates()

In [6]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

df['gender'] = df['gender'].str.title()


In [7]:
# 7. Handle outliers
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

In [8]:
# 8. Feature engineering
df['TenureGroup'] = pd.cut(
    df['tenure'], bins=[-1, 12, 24, 48, 72],
    labels=['New', 'Medium', 'Long', 'Very Long']
)

In [9]:
# 9. Normalize data
scaler = MinMaxScaler()
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[cols] = scaler.fit_transform(df[cols])

In [10]:
# 10. Split dataset
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (5634, 20)
Testing data: (1409, 20)


In [11]:
# 11. Export cleaned dataset
df.to_csv("Cleaned_Telecom_Data.csv", index=False)

print("Cleaned dataset exported successfully!")

# 12. Display final data
print(df.head())
print("Final Shape:", df.shape)

Cleaned dataset exported successfully!
   customerID  gender  SeniorCitizen Partner Dependents    tenure  \
0  7590-VHVEG  Female              0     Yes         No  0.013889   
1  5575-GNVDE    Male              0      No         No  0.472222   
2  3668-QPYBK    Male              0      No         No  0.027778   
3  7795-CFOCW    Male              0      No         No  0.625000   
4  9237-HQITU  Female              0      No         No  0.027778   

  PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0           No  No phone service             DSL             No  ...   
1          Yes                No             DSL            Yes  ...   
2          Yes                No             DSL            Yes  ...   
3           No  No phone service             DSL            Yes  ...   
4          Yes                No     Fiber optic             No  ...   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No          